# Cohort EP2 去識別化資料清理

來源：`Data/cohort_ep2_deid.xlsx` → 輸出：`Data/cohort_clean.csv`、資料字典 `Tables/codebook.csv`

## 三條護欄（全程嚴格遵守）
1. **任何要刪掉的列都要印出刪了哪幾筆、為什麼** —— 絕不默默刪。
2. **離群值只標記、先不刪** —— 以旗標欄位保留，交由分析者判斷。
3. **每一步都印出處理前後的列數，以及該欄的值** —— before / after 全程可追溯。


In [1]:
import os
import pandas as pd
import numpy as np

# 讓 notebook 不論從專案根目錄或 Notebooks/ 執行都能定位資料
if not os.path.exists('Data') and os.path.exists(os.path.join('..', 'Data')):
    os.chdir('..')
print('工作目錄:', os.getcwd())

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

RAW_PATH = 'Data/cohort_ep2_deid.xlsx'

# 全程記錄列數變化，供護欄 3 稽核
row_log = []
def log_rows(step, df):
    row_log.append((step, len(df)))
    print(f'[列數追蹤] {step}: {len(df)} 列')

df = pd.read_excel(RAW_PATH)
N0 = len(df)
log_rows('載入原始資料', df)
print('原始形狀 (列, 欄):', df.shape)
df.head()

工作目錄: /Users/huangshifeng/ep1_stage/Research
[列數追蹤] 載入原始資料: 48 列
原始形狀 (列, 欄): (48, 15)


,Study_ID,性別,年齡,入院日期,出院日期,住院天數,主診斷,糖尿病,高血壓,WBC,Hb,Cr,出院衛教介入,出院前用藥衛教,再入院_30天
0,S001,男,74,2025-03-16,03/24/2025,8,腦中風,0,是,5.9,11.3,2.03,0,NaN,1
1,S002,M,52,01/16/2025,01/31/2025,15,泌尿道感染,是,是,999,10.5,2.47,是,NaN,0
2,S003,男,53,2025/01/29,2025-02-09,11,肺炎,0,是,12.8,11.7,1.02,1,NaN,0
3,S004,F,72,2025-05-03,2025/05/15,12,腦中風,是,1,17.2,10.8,0.98,否,NaN,0
4,S005,女,51,2025-02-28,2025/03/10,10,肺炎,否,是,17.6,10.3,1.81,1,NaN,0


## 0. 原始概觀：欄位與 dtype

In [2]:
print('=== 各欄 dtype ===')
print(df.dtypes)
print('\n=== 欄位清單 ===')
print(list(df.columns))
print('\n=== 數值欄快速描述 ===')
df.describe(include='all').T

=== 各欄 dtype ===
Study_ID     object
性別           object
年齡            int64
入院日期         object
出院日期         object
住院天數          int64
主診斷          object
糖尿病          object
高血壓          object
WBC          object
Hb          float64
Cr           object
出院衛教介入       object
出院前用藥衛教     float64
再入院_30天       int64
dtype: object

=== 欄位清單 ===
['Study_ID', '性別', '年齡', '入院日期', '出院日期', '住院天數', '主診斷', '糖尿病', '高血壓', 'WBC', 'Hb', 'Cr', '出院衛教介入', '出院前用藥衛教', '再入院_30天']

=== 數值欄快速描述 ===


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Study_ID,48,48,S001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
性別,48,4,女,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
年齡,48.0,NaN,NaN,NaN,70.354167,13.191696,48.0,59.25,71.0,81.25,92.0
入院日期,48,48,2025-03-16,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
出院日期,48,47,2024-01-19,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
住院天數,48.0,NaN,NaN,NaN,7.0,3.707382,1.0,4.0,6.0,10.0,15.0
主診斷,48,12,泌尿道感染,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
糖尿病,48.0,4.0,1.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
高血壓,48,4,是,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
WBC,48.0,42.0,999.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. 逐欄 dtype 檢查

辨識「該是數值卻被讀成 object」的欄位（混入文字哨兵值或前後空白）。

In [3]:
print('=== dtype 盤點 ===')
for c in df.columns:
    n_num = pd.to_numeric(df[c], errors='coerce').notna().sum()
    print(f'{c:12s} | dtype={str(df[c].dtype):8s} | 可轉數值={n_num:>2d}/{len(df)} | 範例={df[c].dropna().unique()[:5]}')

# WBC、Cr 名義上是數值，卻為 object → 表示混了文字（哨兵值）或空白字串
print('\n[判讀] WBC、Cr 為 object，混入文字哨兵值/前後空白字串，稍後在「哨兵值」步驟處理。')

=== dtype 盤點 ===
Study_ID     | dtype=object   | 可轉數值= 0/48 | 範例=['S001' 'S002' 'S003' 'S004' 'S005']
性別           | dtype=object   | 可轉數值= 0/48 | 範例=['男' 'M' 'F' '女']
年齡           | dtype=int64    | 可轉數值=48/48 | 範例=[74 52 53 72 51]
入院日期         | dtype=object   | 可轉數值= 0/48 | 範例=['2025-03-16' '01/16/2025' '2025/01/29' '2025-05-03' '2025-02-28']
出院日期         | dtype=object   | 可轉數值= 0/48 | 範例=['03/24/2025' '01/31/2025' '2025-02-09' '2025/05/15' '2025/03/10']
住院天數         | dtype=int64    | 可轉數值=48/48 | 範例=[ 8 15 11 12 10]
主診斷          | dtype=object   | 可轉數值= 0/48 | 範例=['腦中風' '泌尿道感染' '肺炎 ' '肺炎' '心衰竭']
糖尿病          | dtype=object   | 可轉數值=23/48 | 範例=[0 '是' '否' 1]
高血壓          | dtype=object   | 可轉數值=23/48 | 範例=['是' 1 '否' 0]
WBC          | dtype=object   | 可轉數值=46/48 | 範例=[' 5.9 ' 999 12.8 17.2 17.6]
Hb           | dtype=float64  | 可轉數值=48/48 | 範例=[11.3 10.5 11.7 10.8 10.3]
Cr           | dtype=object   | 可轉數值=44/48 | 範例=[2.03 2.47 1.02 0.98 1.81]
出院衛教介入       | dtype=object   | 可轉數值=28/

## 2. 統一日期格式

原始 `入院日期`、`出院日期` 混用三種格式：`YYYY-MM-DD`、`MM/DD/YYYY`、`YYYY/MM/DD`。
全部解析為 `datetime`（供邏輯檢查），最終輸出統一為 ISO `YYYY-MM-DD`。

In [4]:
for col in ['入院日期', '出院日期']:
    before = df[col].copy()
    parsed = pd.to_datetime(df[col], format='mixed', errors='coerce')
    n_fail = parsed.isna().sum()
    print(f'=== {col} ===')
    print('  處理前（原字串範例）:', before.head(6).tolist())
    print('  處理後（datetime）  :', parsed.dt.strftime('%Y-%m-%d').head(6).tolist())
    print(f'  無法解析的筆數: {n_fail}')
    if n_fail:
        print('  解析失敗列:', df.loc[parsed.isna(), 'Study_ID'].tolist())
    df[col] = parsed
    print()
log_rows('統一日期格式後', df)  # 護欄3：日期轉換不刪列，列數不變

=== 入院日期 ===
  處理前（原字串範例）: ['2025-03-16', '01/16/2025', '2025/01/29', '2025-05-03', '2025-02-28', '10/17/2024']
  處理後（datetime）  : ['2025-03-16', '2025-01-16', '2025-01-29', '2025-05-03', '2025-02-28', '2024-10-17']
  無法解析的筆數: 0

=== 出院日期 ===
  處理前（原字串範例）: ['03/24/2025', '01/31/2025', '2025-02-09', '2025/05/15', '2025/03/10', '2024-10-19']
  處理後（datetime）  : ['2025-03-24', '2025-01-31', '2025-02-09', '2025-05-15', '2025-03-10', '2024-10-19']
  無法解析的筆數: 0

[列數追蹤] 統一日期格式後: 48 列


## 3. 合併類別編碼

- **性別**：`男`/`女` 與 `M`/`F` → 統一為 `M`/`F`
- **共病/衛教（是否型）**：`是`/`否` 與 `1`/`0`（int 與 str 混用）→ 統一為 `1`/`0`
- **主診斷**：去除前後空白

In [5]:
# --- 3a. 性別 ---
print('=== 性別 ===')
print('  處理前:', df['性別'].value_counts(dropna=False).to_dict())
sex_map = {'男': 'M', '女': 'F', 'M': 'M', 'F': 'F'}
unmapped = set(df['性別'].dropna().unique()) - set(sex_map)
assert not unmapped, f'性別出現未預期值: {unmapped}'
df['性別'] = df['性別'].map(sex_map)
print('  處理後:', df['性別'].value_counts(dropna=False).to_dict())

=== 性別 ===
  處理前: {'女': 23, '男': 12, 'M': 8, 'F': 5}
  處理後: {'F': 28, 'M': 20}


In [6]:
# --- 3b. 是否型二元欄位：糖尿病 / 高血壓 / 出院衛教介入 ---
bin_map = {'是': 1, '否': 0, 1: 1, 0: 0, '1': 1, '0': 0, 1.0: 1, 0.0: 0}
for col in ['糖尿病', '高血壓', '出院衛教介入']:
    print(f'=== {col} ===')
    print('  處理前:', df[col].value_counts(dropna=False).to_dict())
    unmapped = set(df[col].dropna().unique()) - set(bin_map)
    assert not unmapped, f'{col} 出現未預期值: {unmapped}'
    df[col] = df[col].map(bin_map).astype('Int64')
    print('  處理後:', df[col].value_counts(dropna=False).to_dict())
    print()

=== 糖尿病 ===
  處理前: {1: 14, '否': 13, '是': 12, 0: 9}
  處理後: {np.int64(1): 26, np.int64(0): 22}

=== 高血壓 ===
  處理前: {'是': 20, 1: 16, 0: 7, '否': 5}
  處理後: {np.int64(1): 36, np.int64(0): 12}

=== 出院衛教介入 ===
  處理前: {1: 19, '是': 15, 0: 9, '否': 5}
  處理後: {np.int64(1): 34, np.int64(0): 14}



In [7]:
# --- 3c. 主診斷去前後空白 ---
print('=== 主診斷 ===')
before = sorted(df['主診斷'].unique().tolist())
print('  處理前（注意尾端空白）:', [repr(x) for x in before])
df['主診斷'] = df['主診斷'].str.strip()
after = sorted(df['主診斷'].unique().tolist())
print('  處理後:', after)
print(f'  唯一值數: {len(before)} → {len(after)}（合併了尾端空白造成的假類別）')
log_rows('合併類別編碼後', df)

=== 主診斷 ===
  處理前（注意尾端空白）: ["'心衰竭'", "'心衰竭 '", "'急性腎損傷'", "'慢性阻塞性肺病急性發作'", "'泌尿道感染'", "'糖尿病酮酸中毒'", "'糖尿病酮酸中毒 '", "'肺炎'", "'肺炎 '", "'腦中風'", "'腦中風 '", "'蜂窩性組織炎'"]
  處理後: ['心衰竭', '急性腎損傷', '慢性阻塞性肺病急性發作', '泌尿道感染', '糖尿病酮酸中毒', '肺炎', '腦中風', '蜂窩性組織炎']
  唯一值數: 12 → 8（合併了尾端空白造成的假類別）
[列數追蹤] 合併類別編碼後: 48 列


## 4. 哨兵值轉 NaN

`999`、`未檢`、`未測` 為「未檢驗/缺值」的代碼，非真實量測值，一律轉成 `NaN`。
同時把 `WBC`、`Cr` 去空白後轉為數值型。

In [8]:
SENTINELS = [999, '999', '未檢', '未測', '未驗', '未做']

# --- 4a. WBC：去空白 → 數值，999/未檢 → NaN ---
print('=== WBC ===')
print('  處理前唯一值:', sorted(df['WBC'].astype(str).str.strip().unique(), key=str))
s = df['WBC'].astype(str).str.strip()
mask_sent = s.isin([str(x) for x in SENTINELS])
print(f'  命中哨兵值（999/未檢）筆數: {int(mask_sent.sum())} → 轉 NaN')
df['WBC'] = pd.to_numeric(s.where(~mask_sent, np.nan), errors='coerce')
print('  處理後 dtype:', df['WBC'].dtype, '| 缺失:', int(df['WBC'].isna().sum()), '| 範圍:',
      round(df['WBC'].min(), 1), '-', round(df['WBC'].max(), 1))

=== WBC ===
  處理前唯一值: ['1.8', '10.6', '10.7', '11', '11.1', '11.9', '12.3', '12.6', '12.7', '12.8', '13.2', '14.2', '15.6', '16.3', '17.2', '17.6', '2.8', '3.1', '3.7', '4', '4.2', '4.4', '4.5', '4.8', '4.9', '5.4', '5.7', '5.9', '6.2', '6.8', '6.9', '7.6', '7.7', '8.3', '8.9', '9', '9.3', '9.6', '9.8', '999', '未檢']
  命中哨兵值（999/未檢）筆數: 5 → 轉 NaN
  處理後 dtype: float64 | 缺失: 5 | 範圍: 1.8 - 17.6


In [9]:
# --- 4b. Cr：去空白 → 數值，未測 → NaN ---
print('=== Cr ===')
print('  處理前唯一值:', sorted(df['Cr'].astype(str).str.strip().unique(), key=str))
s = df['Cr'].astype(str).str.strip()
mask_sent = s.isin([str(x) for x in SENTINELS])
print(f'  命中哨兵值（未測）筆數: {int(mask_sent.sum())} → 轉 NaN')
df['Cr'] = pd.to_numeric(s.where(~mask_sent, np.nan), errors='coerce')
print('  處理後 dtype:', df['Cr'].dtype, '| 缺失:', int(df['Cr'].isna().sum()), '| 範圍:',
      round(df['Cr'].min(), 2), '-', round(df['Cr'].max(), 2))

=== Cr ===
  處理前唯一值: ['0.46', '0.53', '0.54', '0.55', '0.76', '0.92', '0.98', '0.99', '1.02', '1.05', '1.09', '1.2', '1.21', '1.32', '1.48', '1.58', '1.59', '1.62', '1.68', '1.69', '1.71', '1.81', '1.84', '1.9', '1.94', '1.97', '2.03', '2.05', '2.06', '2.11', '2.17', '2.19', '2.22', '2.29', '2.33', '2.47', '2.61', '2.65', '2.81', '未測']
  命中哨兵值（未測）筆數: 4 → 轉 NaN
  處理後 dtype: float64 | 缺失: 4 | 範圍: 0.46 - 2.81


In [10]:
# --- 4c. Hb：999 為哨兵值（正常 Hb 約 6-18 g/dL）→ NaN ---
print('=== Hb ===')
print('  處理前範圍:', df['Hb'].min(), '-', df['Hb'].max(), '| 可疑高值:', sorted(df.loc[df['Hb'] > 100, 'Hb'].unique().tolist()))
mask_sent = df['Hb'].isin([999, 999.0])
print(f'  命中哨兵值（999）筆數: {int(mask_sent.sum())} → 轉 NaN ｜ 對應列: {df.loc[mask_sent, "Study_ID"].tolist()}')
df.loc[mask_sent, 'Hb'] = np.nan
print('  處理後範圍:', df['Hb'].min(), '-', df['Hb'].max(), '| 缺失:', int(df['Hb'].isna().sum()))
log_rows('哨兵值轉 NaN 後', df)  # 護欄1/3：哨兵值是「轉 NaN」非「刪列」，列數不變

=== Hb ===
  處理前範圍: 6.3 - 999.0 | 可疑高值: [999.0]
  命中哨兵值（999）筆數: 4 → 轉 NaN ｜ 對應列: ['S025', 'S033', 'S040', 'S046']
  處理後範圍: 6.3 - 16.3 | 缺失: 4
[列數追蹤] 哨兵值轉 NaN 後: 48 列


## 5. 合理性與跨欄邏輯檢查

- 出院日期 **不早於** 入院日期
- 住院天數 **對得上** `(出院 - 入院)` 的天數
- 年齡、住院天數落在合理範圍

> 護欄 1：本步驟若發現邏輯矛盾的列，**只列出、不自動刪**，並印出原因。

In [11]:
issues = pd.DataFrame({'Study_ID': df['Study_ID']})

# 5a. 出院 >= 入院
bad_order = df['出院日期'] < df['入院日期']
issues['出院早於入院'] = bad_order

# 5b. 住院天數 == (出院 - 入院).days
los_calc = (df['出院日期'] - df['入院日期']).dt.days
mismatch = los_calc.notna() & (los_calc != df['住院天數'])
issues['住院天數不符'] = mismatch

# 5c. 合理範圍
issues['年齡超出範圍'] = ~df['年齡'].between(0, 120)
issues['住院天數非正'] = df['住院天數'] <= 0

print('=== 跨欄/合理性檢查結果（每項違規筆數）===')
for c in issues.columns[1:]:
    print(f'  {c}: {int(issues[c].sum())} 筆')

flagged = issues[issues.iloc[:, 1:].any(axis=1)]
print(f'\n=== 有任一邏輯問題的列：{len(flagged)} 筆 ===')
if len(flagged):
    print(flagged.to_string(index=False))
    print('\n[護欄1] 偵測到邏輯矛盾，已列出上述列，等待人工判斷，本步驟不自動刪除。')
else:
    print('全部通過 → 沒有任何列因邏輯矛盾需要刪除。')

=== 跨欄/合理性檢查結果（每項違規筆數）===
  出院早於入院: 0 筆
  住院天數不符: 0 筆
  年齡超出範圍: 0 筆
  住院天數非正: 0 筆

=== 有任一邏輯問題的列：0 筆 ===
全部通過 → 沒有任何列因邏輯矛盾需要刪除。


## 6. 列刪除決策（護欄 1）

可能的刪列理由：重複 `Study_ID`、整列重複、關鍵欄位（ID/日期）缺失、或上一步的邏輯矛盾。
**逐項列出命中的列與原因；沒有命中就保留全部。**

In [12]:
drop_reasons = {}

# 6a. 重複 Study_ID
dup_id = df['Study_ID'].duplicated(keep='first')
if dup_id.any():
    drop_reasons['重複 Study_ID'] = df.loc[dup_id, 'Study_ID'].tolist()

# 6b. 整列重複
dup_row = df.duplicated(keep='first')
if dup_row.any():
    drop_reasons['整列完全重複'] = df.loc[dup_row, 'Study_ID'].tolist()

# 6c. 關鍵欄位缺失（ID 或兩個日期皆無法解析）
key_missing = df['Study_ID'].isna() | (df['入院日期'].isna() & df['出院日期'].isna())
if key_missing.any():
    drop_reasons['關鍵欄位缺失'] = df.loc[key_missing, 'Study_ID'].tolist()

print('=== 刪列稽核 ===')
print(f'  處理前列數: {len(df)}')
if drop_reasons:
    to_drop = set()
    for reason, ids in drop_reasons.items():
        print(f'  ✗ 因「{reason}」刪除: {ids}')
        to_drop |= set(ids)
    df = df[~df['Study_ID'].isin(to_drop)].reset_index(drop=True)
    print(f'  共刪除 {len(to_drop)} 筆')
else:
    print('  ✓ 無重複、無整列重複、無關鍵欄位缺失 → 刪除 0 筆，保留全部。')
print(f'  處理後列數: {len(df)}')
log_rows('列刪除決策後', df)

=== 刪列稽核 ===
  處理前列數: 48
  ✓ 無重複、無整列重複、無關鍵欄位缺失 → 刪除 0 筆，保留全部。
  處理後列數: 48
[列數追蹤] 列刪除決策後: 48 列


## 7. 量化各欄缺失並決定處理方式

對每欄計算缺失數與比例，依比例給出處理建議：
- 缺失極高（>90%）→ 資訊量幾近於零，**移除整欄**
- 中低度缺失 → **保留並標記**（含哨兵轉換後的 NaN），交由分析時以統計方法處理，不臆造數值

In [13]:
miss = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(1),
})
miss['decision'] = np.where(miss['missing_pct'] > 90, '移除整欄（資訊量過低）',
                     np.where(miss['missing_pct'] > 0, '保留並標記（不填補）', '無缺失'))
print('=== 各欄缺失量化與處理決策 ===')
print(miss.to_string())

drop_cols = miss.index[miss['missing_pct'] > 90].tolist()
print(f'\n[決策] 移除欄位: {drop_cols}')
if drop_cols:
    for c in drop_cols:
        print(f'  ✗ {c}: 缺失 {miss.loc[c, "n_missing"]}/{len(df)} ({miss.loc[c, "missing_pct"]}%) → 移除')
    df = df.drop(columns=drop_cols)
print(f'剩餘欄位數: {df.shape[1]}')
log_rows('缺失處理後（注意：移除的是欄不是列）', df)

=== 各欄缺失量化與處理決策 ===
          n_missing  missing_pct     decision
Study_ID          0          0.0          無缺失
性別                0          0.0          無缺失
年齡                0          0.0          無缺失
入院日期              0          0.0          無缺失
出院日期              0          0.0          無缺失
住院天數              0          0.0          無缺失
主診斷               0          0.0          無缺失
糖尿病               0          0.0          無缺失
高血壓               0          0.0          無缺失
WBC               5         10.4   保留並標記（不填補）
Hb                4          8.3   保留並標記（不填補）
Cr                4          8.3   保留並標記（不填補）
出院衛教介入            0          0.0          無缺失
出院前用藥衛教          46         95.8  移除整欄（資訊量過低）
再入院_30天           0          0.0          無缺失

[決策] 移除欄位: ['出院前用藥衛教']
  ✗ 出院前用藥衛教: 缺失 46/48 (95.8%) → 移除
剩餘欄位數: 14
[列數追蹤] 缺失處理後（注意：移除的是欄不是列）: 48 列


## 8. 衍生欄位

- **leukocytosis（白血球增多）**：`WBC > 11.0`（×10⁹/L，臨床常用上限）→ 旗標 1/0；WBC 缺失則維持缺失。
- **comorbidity_count（共病數）**：`糖尿病 + 高血壓` → 0/1/2。

In [14]:
# 8a. 白血球增多
WBC_HIGH = 11.0
df['leukocytosis'] = np.where(df['WBC'].isna(), pd.NA, (df['WBC'] > WBC_HIGH).astype('Int64'))
df['leukocytosis'] = df['leukocytosis'].astype('Int64')
print(f'=== leukocytosis (WBC > {WBC_HIGH}) ===')
print('  分布:', df['leukocytosis'].value_counts(dropna=False).to_dict())

# 8b. 共病數
df['comorbidity_count'] = (df['糖尿病'].fillna(0) + df['高血壓'].fillna(0)).astype('Int64')
print('\n=== comorbidity_count (糖尿病 + 高血壓) ===')
print('  分布:', df['comorbidity_count'].value_counts(dropna=False).sort_index().to_dict())
log_rows('新增衍生欄位後', df)

=== leukocytosis (WBC > 11.0) ===
  分布: {np.int64(0): 29, np.int64(1): 14, <NA>: 5}

=== comorbidity_count (糖尿病 + 高血壓) ===
  分布: {np.int64(0): 7, np.int64(1): 20, np.int64(2): 21}
[列數追蹤] 新增衍生欄位後: 48 列


## 9. 欄位改為英文 snake_case

In [15]:
rename_map = {
    'Study_ID': 'study_id',
    '性別': 'sex',
    '年齡': 'age',
    '入院日期': 'admission_date',
    '出院日期': 'discharge_date',
    '住院天數': 'length_of_stay',
    '主診斷': 'primary_diagnosis',
    '糖尿病': 'diabetes',
    '高血壓': 'hypertension',
    'WBC': 'wbc',
    'Hb': 'hb',
    'Cr': 'cr',
    '出院衛教介入': 'discharge_education',
    '再入院_30天': 'readmission_30d',
}
print('=== 欄位改名 ===')
for old, new in rename_map.items():
    if old in df.columns:
        print(f'  {old:12s} → {new}')
df = df.rename(columns=rename_map)
print('\n最終欄位:', list(df.columns))

=== 欄位改名 ===
  Study_ID     → study_id
  性別           → sex
  年齡           → age
  入院日期         → admission_date
  出院日期         → discharge_date
  住院天數         → length_of_stay
  主診斷          → primary_diagnosis
  糖尿病          → diabetes
  高血壓          → hypertension
  WBC          → wbc
  Hb           → hb
  Cr           → cr
  出院衛教介入       → discharge_education
  再入院_30天      → readmission_30d

最終欄位: ['study_id', 'sex', 'age', 'admission_date', 'discharge_date', 'length_of_stay', 'primary_diagnosis', 'diabetes', 'hypertension', 'wbc', 'hb', 'cr', 'discharge_education', 'readmission_30d', 'leukocytosis', 'comorbidity_count']


## 10. 離群值標記（護欄 2：只標、不刪）

對連續欄位（`age`、`length_of_stay`、`wbc`、`hb`、`cr`）以 IQR 法（1.5×IQR）標記離群值，
新增 `*_outlier` 布林旗標欄位**保留於輸出**，供分析者判斷，**不刪除任何列**。

In [16]:
def iqr_flags(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (series < lo) | (series > hi), lo, hi

num_cols = ['age', 'length_of_stay', 'wbc', 'hb', 'cr']
print('=== 離群值標記（IQR 1.5；只標記不刪）===')
for c in num_cols:
    flag, lo, hi = iqr_flags(df[c])
    flag = flag.fillna(False)
    df[c + '_outlier'] = flag
    ids = df.loc[flag, 'study_id'].tolist()
    print(f'  {c:15s} 正常區間 [{lo:.2f}, {hi:.2f}] | 離群 {int(flag.sum())} 筆 {ids}')

any_out = df[[c + '_outlier' for c in num_cols]].any(axis=1)
print(f'\n至少一項離群的列: {int(any_out.sum())} 筆（全部保留，僅旗標標記）')
log_rows('離群值標記後（不刪列）', df)

=== 離群值標記（IQR 1.5；只標記不刪）===
  age             正常區間 [26.25, 114.25] | 離群 0 筆 []
  length_of_stay  正常區間 [-5.00, 19.00] | 離群 0 筆 []
  wbc             正常區間 [-5.58, 23.03] | 離群 0 筆 []
  hb              正常區間 [6.51, 16.61] | 離群 1 筆 ['S017']
  cr              正常區間 [-0.41, 3.56] | 離群 0 筆 []

至少一項離群的列: 1 筆（全部保留，僅旗標標記）
[列數追蹤] 離群值標記後（不刪列）: 48 列


## 11. 最終彙整與輸出

確認最終 dtype、列數，輸出統一日期格式的 `Data/cohort_clean.csv`。

In [17]:
# 日期統一輸出為 ISO 字串 YYYY-MM-DD
out = df.copy()
for c in ['admission_date', 'discharge_date']:
    out[c] = out[c].dt.strftime('%Y-%m-%d')

print('=== 最終 dtype ===')
print(out.dtypes)
print('\n=== 列數變化全程追蹤（護欄 3）===')
for step, n in row_log:
    print(f'  {step:32s}: {n} 列')
print(f'\n原始 {N0} 列 → 最終 {len(out)} 列（淨刪除 {N0 - len(out)} 列）')

OUT_PATH = 'Data/cohort_clean.csv'
out.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(f'\n✓ 已輸出 {OUT_PATH}  形狀={out.shape}')
out.head()

=== 最終 dtype ===
study_id                   object
sex                        object
age                         int64
admission_date             object
discharge_date             object
length_of_stay              int64
primary_diagnosis          object
diabetes                    Int64
hypertension                Int64
wbc                       float64
hb                        float64
cr                        float64
discharge_education         Int64
readmission_30d             int64
leukocytosis                Int64
comorbidity_count           Int64
age_outlier                  bool
length_of_stay_outlier       bool
wbc_outlier                  bool
hb_outlier                   bool
cr_outlier                   bool
dtype: object

=== 列數變化全程追蹤（護欄 3）===
  載入原始資料                          : 48 列
  統一日期格式後                         : 48 列
  合併類別編碼後                         : 48 列
  哨兵值轉 NaN 後                      : 48 列
  列刪除決策後                          : 48 列
  缺失處理後（注意：移除的是欄不是列）       

,study_id,sex,age,admission_date,discharge_date,length_of_stay,primary_diagnosis,diabetes,hypertension,wbc,hb,cr,discharge_education,readmission_30d,leukocytosis,comorbidity_count,age_outlier,length_of_stay_outlier,wbc_outlier,hb_outlier,cr_outlier
0,S001,M,74,2025-03-16,2025-03-24,8,腦中風,0,1,5.9,11.3,2.03,0,1,0,1,False,False,False,False,False
1,S002,M,52,2025-01-16,2025-01-31,15,泌尿道感染,1,1,NaN,10.5,2.47,1,0,<NA>,2,False,False,False,False,False
2,S003,M,53,2025-01-29,2025-02-09,11,肺炎,0,1,12.8,11.7,1.02,1,0,1,1,False,False,False,False,False
3,S004,F,72,2025-05-03,2025-05-15,12,腦中風,1,1,17.2,10.8,0.98,0,0,1,2,False,False,False,False,False
4,S005,F,51,2025-02-28,2025-03-10,10,肺炎,0,1,17.6,10.3,1.81,1,0,1,1,False,False,False,False,False


## 12. 產生 codebook 資料字典

逐欄記錄：英文變數名、中文標籤、型別、單位、允許值/範圍、缺失數/比例、衍生來源與備註。
輸出至 `Tables/codebook.csv`。

In [18]:
meta = {
 'study_id':          ('研究編號（去識別化）', '識別碼', '', '唯一鍵'),
 'sex':               ('性別', '類別', '', 'M=男, F=女；由 男/女/M/F 統一'),
 'age':               ('年齡', '數值', '歲', '入院當下年齡'),
 'admission_date':    ('入院日期', '日期', 'YYYY-MM-DD', '由三種混合格式統一'),
 'discharge_date':    ('出院日期', '日期', 'YYYY-MM-DD', '由三種混合格式統一；不早於入院'),
 'length_of_stay':    ('住院天數', '數值', '天', '與(出院-入院)一致性已驗證'),
 'primary_diagnosis': ('主診斷', '類別', '', '已去前後空白'),
 'diabetes':          ('糖尿病共病', '二元', '', '1=是, 0=否；由 是否/1/0 統一'),
 'hypertension':      ('高血壓共病', '二元', '', '1=是, 0=否；由 是否/1/0 統一'),
 'wbc':               ('白血球數', '數值', '10^9/L', '999/未檢 已轉 NaN'),
 'hb':                ('血色素', '數值', 'g/dL', '999 哨兵值已轉 NaN'),
 'cr':                ('肌酸酐', '數值', 'mg/dL', '未測 已轉 NaN'),
 'discharge_education':('出院衛教介入', '二元', '', '1=有, 0=無；由 是否/1/0 統一'),
 'readmission_30d':   ('30 天內再入院', '二元', '', '1=是, 0=否'),
 'leukocytosis':      ('白血球增多旗標', '二元(衍生)', '', '衍生：wbc > 11.0；wbc 缺失則為 NaN'),
 'comorbidity_count': ('共病數', '數值(衍生)', '', '衍生：diabetes + hypertension (0-2)'),
}

rows = []
for col in out.columns:
    s = out[col]
    if col.endswith('_outlier'):
        label = f'{col.replace("_outlier", "")} 離群旗標'
        vtype, unit, note = '二元(衍生)', '', 'IQR 1.5 標記，僅標記未刪除（護欄2）'
        allowed = 'True/False'
    else:
        label, vtype, unit, note = meta.get(col, (col, '', '', ''))
        if s.dtype == object or vtype.startswith('類別') or vtype.startswith('識別'):
            uniq = s.dropna().unique().tolist()
            allowed = '; '.join(map(str, uniq[:12])) + (' ...' if len(uniq) > 12 else '')
        elif '二元' in vtype:
            allowed = '0/1' if set(s.dropna().unique()) <= {0, 1} else str(sorted(s.dropna().unique().tolist()))
        elif '日期' in vtype:
            allowed = f'{s.min()} ~ {s.max()}'
        else:
            num = pd.to_numeric(s, errors='coerce')
            allowed = f'{num.min()} ~ {num.max()}'
    rows.append({
        'variable': col, 'label': label, 'type': vtype, 'unit': unit,
        'allowed_values_or_range': allowed,
        'n_missing': int(s.isna().sum()),
        'missing_pct': round(s.isna().mean() * 100, 1),
        'notes': note,
    })

codebook = pd.DataFrame(rows)
CB_PATH = 'Tables/codebook.csv'
codebook.to_csv(CB_PATH, index=False, encoding='utf-8-sig')
print(f'✓ 已輸出 {CB_PATH}  （{len(codebook)} 個變數）')
codebook

✓ 已輸出 Tables/codebook.csv  （21 個變數）


,variable,label,type,unit,allowed_values_or_range,n_missing,missing_pct,notes
0,study_id,研究編號（去識別化）,識別碼,,S001; S002; S003; S004; S005; S006; S007; S008...,0,0.0,唯一鍵
1,sex,性別,類別,,M; F,0,0.0,"M=男, F=女；由 男/女/M/F 統一"
2,age,年齡,數值,歲,48 ~ 92,0,0.0,入院當下年齡
3,admission_date,入院日期,日期,YYYY-MM-DD,2025-03-16; 2025-01-16; 2025-01-29; 2025-05-03...,0,0.0,由三種混合格式統一
4,discharge_date,出院日期,日期,YYYY-MM-DD,2025-03-24; 2025-01-31; 2025-02-09; 2025-05-15...,0,0.0,由三種混合格式統一；不早於入院
5,length_of_stay,住院天數,數值,天,1 ~ 15,0,0.0,與(出院-入院)一致性已驗證
6,primary_diagnosis,主診斷,類別,,腦中風; 泌尿道感染; 肺炎; 心衰竭; 糖尿病酮酸中毒; 急性腎損傷; 蜂窩性組織炎; 慢...,0,0.0,已去前後空白
7,diabetes,糖尿病共病,二元,,0/1,0,0.0,"1=是, 0=否；由 是否/1/0 統一"
8,hypertension,高血壓共病,二元,,0/1,0,0.0,"1=是, 0=否；由 是否/1/0 統一"
9,wbc,白血球數,數值,10^9/L,1.8 ~ 17.6,5,10.4,999/未檢 已轉 NaN
